# 2 · Without an experiment: confounding and its correction

We turn the RCT into observational data: high-value, recently active customers are more likely to receive the e-mail.
Units are subsampled with P(kept | X) constant, so the covariate distribution and the true ATE stay those of the RCT while treatment now depends on X.

**Identifying assumptions** (what makes the correction work):
1. *No unobserved confounding*: every common cause of mailing and visiting is in X. True here by construction, never testable in real data.
2. *Overlap*: every customer has a propensity strictly between 0 and 1 (clipped to [0.05, 0.95]).
3. *SUTVA*: one customer's e-mail does not affect another's visit.

In [1]:
import pandas as pd

from uplift_causal.data import load_hillstrom
from uplift_causal.estimate import fit_estimators, refute, summarize
from uplift_causal.evaluate_uplift import diff_means
from uplift_causal.plots import dag_plot, estimate_plot, save
from uplift_causal.simulate_bias import simulate_bias

WEIGHTS = {"log_history": 1.0, "recency": -1.0}

/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rct = load_hillstrom()
obs = simulate_bias(rct, WEIGHTS, strength=1.5)
truth = diff_means(rct.y, rct.t)
len(rct), len(obs), obs.t.mean()

In [3]:
save(dag_plot(["log_history", "recency"], ["channel", "zip_code", "newbie"]), "dag", height=420)

In [4]:
X = obs.drop(columns=["t", "y"])
smd = (X[obs.t == 1].mean() - X[obs.t == 0].mean()) / X.std()
smd.sort_values(key=abs, ascending=False).round(3).to_frame("standardized mean difference")

,standardized mean difference
recency,-1.050
log_history,1.007
channel_Web,-0.140
channel_Phone,-0.137
womens,0.117
mens,0.106
newbie,0.106
zip_code_Surburban,-0.023
zip_code_Urban,0.005


In [5]:
estimates = fit_estimators(obs, n_boot=100)
summary = pd.concat([
    pd.DataFrame([diff_means(obs.y, obs.t)], index=["Naive"], columns=["est", "lo", "hi"]),
    summarize(estimates),
])
summary.assign(bias=summary.est - truth[0]).round(4)

,est,lo,hi,bias
Naive,0.0868,0.0779,0.0957,0.0415
PS Matching,0.0551,0.0416,0.0700,0.0099
IPW,0.0531,0.0393,0.0647,0.0079
DML,0.0480,0.0360,0.0600,0.0027


In [6]:
save(estimate_plot(summary, truth), "hero", height=380)

## Refutation tests
A random extra confounder should leave the estimate unchanged, a permuted placebo treatment should drive it to zero, and re-estimating on subsets should reproduce it.

In [7]:
refute(obs, "IPW").round(4)

,estimate,refuted,p_value
random_common_cause,0.0531,0.0531,0.4562
placebo_treatment_refuter,0.0531,-0.0002,0.4788
data_subset_refuter,0.0531,0.0528,0.4709
